In [ ]:
%pip install ultralytics


In [ ]:
import os
import sys
import math
import cv2
import numpy as np
import pandas as pd
from ultralytics import YOLO
import importlib
from pathlib import Path

# BASE PROJECT PATH
BASE_DIR = "/Users/florawang/Downloads/Rijo-Ferreira Lab"
PROJECT_DIR = BASE_DIR  # since utils/ is already here

OG_PROJECT_DIR = os.path.join(
    BASE_DIR,
    "MosquitoMovement2"
)

sys.path.append(PROJECT_DIR)

from utils.image_util import *

_NB_ROOT = Path.cwd()
if (_NB_ROOT / "mosquito_lab").is_dir():
    sys.path.insert(0, str(_NB_ROOT))
elif (_NB_ROOT.parent / "mosquito-lab").is_dir():
    sys.path.insert(0, str(_NB_ROOT.parent / "mosquito-lab"))

import mosquito_lab.status as inference_status
importlib.reload(inference_status)
from mosquito_lab.status import write_status

# NEW DATA PATHs
LABEL_FILE = "/Users/florawang/Downloads/Rijo-Ferreira Lab/MosquitoMovement2/experiments/29 - 20260617_ClockKO_rep5_box1/exp29_labels.csv"

MODEL_PATH = "/Users/florawang/Downloads/Rijo-Ferreira Lab/MosquitoMovement2/models/uninf_det_v0.pt"

IMAGE_FOLDER = "/Users/florawang/Downloads/Rijo-Ferreira Lab/data/29 - 20260617_ClockKO_rep5_box1/[29] raw_images"

OUTPUT_FOLDER = "/Users/florawang/Downloads/Rijo-Ferreira Lab/MosquitoMovement2/analysis/results"

os.makedirs(OUTPUT_FOLDER, exist_ok=True)

OUTPUT_FILE = os.path.join(
    OUTPUT_FOLDER,
    "29_20260617_ClockK_rep5_box1_transposed_2.csv",
)

STATUS_FILE = os.path.join(OUTPUT_FOLDER, "inference_status.json")

# SETTINGS

# Assumes 1 frame = 1 minute
FRAME_TO_MINUTES = 1

# Starting ZT
START_ZT = 0

# DEBUG PRINTS

print("\n========== PATH CHECK ==========")

print("BASE_DIR:")
print(BASE_DIR)

print("\nMODEL EXISTS:")
print(os.path.exists(MODEL_PATH))

print("\nIMAGE FOLDER EXISTS:")
print(os.path.exists(IMAGE_FOLDER))

print("\nLABEL FILE EXISTS:")
print(os.path.exists(LABEL_FILE))

print("\n================================\n")


# Initialization
print("Loading cropper...")
cropper = Cropper(LABEL_FILE)

print("Loading YOLO model...")
model = YOLO(MODEL_PATH)

# force CPU
model.to("cpu")

print("YOLO loaded.\n")

# GLOBAL STATS
GLOBAL_STATS = {
    "failed_loads": 0,
    "failed_crops": 0,
    "failed_detections": 0,
    "successful_detections": 0
}


# SAFE CROP
def safe_crop(image, idx):

    try:

        cropped = cropper.crop(image, idx)

        if cropped is None:
            return None

        if len(cropped.shape) < 2:
            return None

        h, w = cropped.shape[:2]

        if h == 0 or w == 0:
            return None

        return cropped

    except Exception as e:

        print(f"Crop error mosquito {idx}: {e}")
        return None


# DETECT POSITION
def detect_pos_from_crop(cropped):

    try:

        result = model(cropped, verbose=False)

        if len(result[0].boxes) == 0:
            return None

        pos = result[0].boxes[0].xywh[0].cpu().numpy()

        center_x = pos[0] + pos[2] / 2
        center_y = pos[1] + pos[3] / 2

        return (center_x, center_y)

    except Exception as e:

        print("Detection error:", e)
        return None


# PROCESS SINGLE MOSQUITO
def process_image(image_paths, mosquito_num, mosquito_total):

    write_status(
        STATUS_FILE,
        state="running",
        mosquito_num=mosquito_num,
        mosquito_total=mosquito_total,
        current_photo=0,
        total_photos=len(image_paths),
    )

    activity = []

    total_frames = 0
    failed_loads = 0
    failed_crops = 0
    failed_detections = 0
    successful_detections = 0

    last_pos = None

    for i, path in enumerate(image_paths):

        total_frames += 1

        if i % 100 == 0:

            print(
                f"[Mosquito {mosquito_num}] "
                f"{i}/{len(image_paths)}"
            )

            write_status(
                STATUS_FILE,
                state="running",
                mosquito_num=mosquito_num,
                mosquito_total=mosquito_total,
                current_photo=i,
                total_photos=len(image_paths),
            )

        full_path = os.path.join(
            IMAGE_FOLDER,
            path
        )


        # LOAD IMAGE
        frame = cv2.imread(full_path)

        if frame is None:

            failed_loads += 1
            activity.append(np.nan)
            continue

        # CROP
        cropped = safe_crop(frame, mosquito_num)

        if cropped is None:

            failed_crops += 1
            activity.append(np.nan)
            continue

        # DETECT MOSQUITO
        pos = detect_pos_from_crop(cropped)

        if pos is None:

            failed_detections += 1
            activity.append(np.nan)
            continue

        successful_detections += 1


        # DISTANCE MOVED
        if last_pos is None:
            dist = 0

        else:
            dist = math.sqrt(
                (last_pos[0] - pos[0]) ** 2 +
                (last_pos[1] - pos[1]) ** 2
            )
        last_pos = pos
        activity.append(dist)

    
    # PRINT STATS
    print(f"\nMosquito {mosquito_num} Summary")

    print("Frames:", total_frames)
    print("Failed loads:", failed_loads)
    print("Failed crops:", failed_crops)
    print("Failed detections:", failed_detections)
    print("Successful detections:", successful_detections)

    GLOBAL_STATS["failed_loads"] += failed_loads
    GLOBAL_STATS["failed_crops"] += failed_crops
    GLOBAL_STATS["failed_detections"] += failed_detections
    GLOBAL_STATS["successful_detections"] += successful_detections

    return activity

# MAIN
if __name__ == "__main__":

    try:
        mosquito_total = len(cropper.boxes)

        print("Starting inference...\n")
        print(f"Progress is written to: {STATUS_FILE}\n")

        # GET IMAGE PATHS
        image_paths = sorted(
            [
                f for f in os.listdir(IMAGE_FOLDER)
                if f.lower().endswith(
                    (".jpg", ".jpeg", ".png")
                )
            ],
            key=lambda x: float(x.split(".")[0])
        )

        print("Images found:", len(image_paths))

        write_status(
            STATUS_FILE,
            state="running",
            mosquito_num=0,
            mosquito_total=mosquito_total,
            current_photo=0,
            total_photos=len(image_paths),
        )

        # STORE ALL MOSQUITO DATA
        all_data = {}

        # PROCESS EACH MOSQUITO
        for mosquito_num, _ in enumerate(cropper.boxes):

            print("\n===================================")
            print(f"PROCESSING MOSQUITO {mosquito_num}")
            print("===================================\n")

            activity = process_image(
                image_paths,
                mosquito_num,
                mosquito_total,
            )

            # each mosquito becomes one column
            all_data[
                f"mosquito_{mosquito_num}"
            ] = activity

        # CREATE DATAFRAME
        print("\nCreating dataframe...")

        df = pd.DataFrame(all_data)

        # ADD FRAME COLUMN
        df.insert(
            0,
            "frame",
            range(len(df))
        )

        # SAVE CSV
        print("\nSaving CSV...")

        df.to_csv(
            OUTPUT_FILE,
            index=False
        )

        print("\n===================================")
        print("DONE")
        print("===================================\n")

        print("Saved CSV:")
        print(OUTPUT_FILE)

        print("\nGLOBAL STATS:")
        print(GLOBAL_STATS)

        print(MODEL_PATH)

        write_status(
            STATUS_FILE,
            state="done",
            mosquito_num=mosquito_total - 1 if mosquito_total else None,
            mosquito_total=mosquito_total,
            current_photo=len(image_paths),
            total_photos=len(image_paths),
            output_file=OUTPUT_FILE,
        )

    except Exception as exc:
        write_status(STATUS_FILE, state="failed", error=f"{type(exc).__name__}: {exc}")
        raise

print(os.path.exists(MODEL_PATH))


Checking # of images

In [ ]:
all_files = os.listdir(IMAGE_FOLDER)

print("TOTAL FILES IN FOLDER:", len(all_files))

valid_images = [
    f for f in all_files
    if f.lower().endswith((".jpg", ".jpeg", ".png"))
]

print("VALID IMAGE FILES:", len(valid_images))

missing = set(all_files) - set(valid_images)

print("\nNON-IMAGE FILES:")
for m in sorted(missing)[:20]:
    print(m)

TOTAL FILES IN FOLDER: 6574
VALID IMAGE FILES: 6573

NON-IMAGE FILES:
data_collection.csv
